<a href="https://colab.research.google.com/github/Skylight2703/Code-Explanation-Tutor/blob/main/code_explaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Ollama CLI
!curl -fsSL https://ollama.com/install.sh | sh

# Install required Python packages
!pip install pypdf requests

import subprocess
import time

# Start the Ollama service in the background
ollama_process = subprocess.Popen(["ollama", "serve"])

# Wait a few seconds for the service to start up
time.sleep(5)
print("Ollama server is up and running!")

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 25.1 MB/s eta 0:00:00


FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

In [2]:
# 1. Install missing system dependencies (zstd is required for Ollama extraction)
!apt-get update && apt-get install -y zstd

# 2. Install Ollama CLI
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Install Python packages
!pip install pypdf requests

import subprocess
import time

# 4. Start the Ollama service in the background
ollama_process = subprocess.Popen(["ollama", "serve"])

# Wait a few seconds for the service to start up
time.sleep(5)
print("Ollama server is up and running successfully!")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [359 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:10 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1,801 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:13 http://archive.ubuntu.com/ubuntu noble-backports InRelease 

In [3]:
!ollama pull llama3.2

In [4]:
import os
import requests
from pypdf import PdfReader

# 1. File Reader Function
def get_code_from_file(file_path):
    """Extracts text content from a text, code, or PDF file."""
    if not os.path.exists(file_path):
        return f"Error: File '{file_path}' does not exist."

    file_extension = os.path.splitext(file_path)[1].lower()

    if file_extension == ".pdf":
        try:
            reader = PdfReader(file_path)
            full_text = ""
            for page in reader.pages:
                full_text += page.extract_text() + "\n"
            return full_text
        except Exception as e:
            return f"Error reading PDF: {e}"
    else:
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                return file.read()
        except UnicodeDecodeError:
            with open(file_path, "r", encoding="latin-1") as file:
                return file.read()
        except Exception as e:
            return f"Error reading file: {e}"


# 2. Ollama API Caller
def generate_llm_response(prompt, model_name="llama3.2"):
    """Sends a prompt to the locally running Ollama API and returns the response."""
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False
    }

    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        return response.json().get("response", "")
    except Exception as e:
        return f"Error contacting Ollama API: {e}"


# 3. RAG Context Integration Function
def answer_user_query(retrieved_context, user_question, model_name="llama3.2"):
    """
    Combines retrieved context from your friend's RAG pipeline with the user's question.
    """
    prompt = f"""
    You are an expert software developer and code assistant.
    Answer the user's question based strictly on the provided code context.
    If the code context does not contain enough information to answer the question, state that clearly.

    --- CODE CONTEXT ---
    {retrieved_context}

    --- USER QUESTION ---
    {user_question}

    --- ANSWER ---
    """
    return generate_llm_response(prompt, model_name=model_name)


# --- Test Run ---
# Create a dummy code file to test the full pipeline
sample_code = """
def calculate_factorial(n):
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    if n in (0, 1):
        return 1
    return n * calculate_factorial(n - 1)
"""

with open("example.py", "w") as f:
    f.write(sample_code)

# Load code and query LLM
extracted_code = get_code_from_file("example.py")
user_query = "What happens if a negative number is passed to calculate_factorial?"

print("Querying Ollama...\n")
response = answer_user_query(retrieved_context=extracted_code, user_question=user_query)

print("--- Question ---")
print(user_query)
print("\n--- LLM Response ---")
print(response)

Querying Ollama...

--- Question ---
What happens if a negative number is passed to calculate_factorial?

--- LLM Response ---
The code raises a ValueError with the message "Factorial is not defined for negative numbers." when a negative number is passed to the calculate_factorial function.
